# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following the Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: {}".format(metadata.name))
print("Description:\n{}".format(metadata.description))

## 2. Data Overview
Review available record sets, fields, and their `@id` values. These identifiers are important for precise referencing and data extraction.


In [ ]:
# Explore the record sets defined in the dataset
print("All record sets found in the dataset:\n")
for record_set in metadata.record_sets:
    print(f"- Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description if hasattr(record_set, 'description') else ''}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for f in record_set.fields:
            print(f"    - {getattr(f, 'name', '')} (@id: {getattr(f, 'id', f)})")
    print()

### Example: Display first few records from a record set
Let's preview records from the first available record set as an example. You can adjust the `record_set_id` to select others.


In [ ]:
# Select the record set @id to preview (update if your dataset has multiple record sets)
record_set_ids = [rs.id for rs in metadata.record_sets]
if not record_set_ids:
    raise RuntimeError('No record sets defined in the dataset schema.')

record_set_id = record_set_ids[0]
print(f"Showing sample records from record set with @id: {record_set_id}\n")

# Print the first 3 records from this record set
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    pprint.pprint(record)
    if i >= 2:
        break

## 3. Data Extraction
Load one or more record sets into pandas DataFrames for data analysis.

Each record set is uniquely identified by its `@id`.

In [ ]:
# Collect data from each record set found in this dataset
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# Display the first few rows of the first DataFrame
first_df = dataframes[record_set_id]
first_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process one of the numeric variables for outlier removal, normalization, and grouping. Make sure to use only the `@id` for selecting fields.

You can adapt the field selection to the columns available in your target record set.

In [ ]:
# Choose a numeric field and a grouping field using their @id values
print('Available columns for EDA:', first_df.columns.tolist())
# Example: Replace with correct @id for your actual numeric field
numeric_field_id = None
group_field_id = None
for col in first_df.columns.tolist():
    # A naive guess: select the first float/integer column as numeric, and first object as group
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(first_df[col]):
        numeric_field_id = col
    if group_field_id is None and not pd.api.types.is_numeric_dtype(first_df[col]):
        group_field_id = col

if not numeric_field_id:
    raise RuntimeError('No numeric field found for EDA. Update numeric_field_id to match your dataset.')
print(f'Using numeric field: {{numeric_field_id}}')

# Filter records based on a threshold (e.g., greater than the median)
threshold = first_df[numeric_field_id].median() if numeric_field_id else 0
filtered_df = first_df[first_df[numeric_field_id] > threshold]
print(f"Filtered records where {{numeric_field_id}} > {{threshold}}. Number of records: {{filtered_df.shape[0]}}")

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"Normalized values for {numeric_field_id} (first few rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by a category field and compute mean of numeric field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (first few rows):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and the grouping results (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))
sns.histplot(first_df[numeric_field_id], kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id in filtered_df.columns and 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to programmatically explore and process the FAIR^2 dataset using the Croissant schema and the `mlcroissant` library.

We:
- Inspected dataset structure using unique @id identifiers
- Loaded records into DataFrames by record set @id
- Filtered, normalized, grouped, and visualized numeric fields for exploratory analysis

**Next steps:**
- Extend the EDA for more fields/record sets as needed
- Apply advanced analysis depending on your research questions
